In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, ClassifierMixin
import joblib
import numpy as np
import warnings
warnings.filterwarnings('ignore')


Some models (like SVM, KNN, and Logistic Regression) perform better with normalized or scaled data.
We define a custom wrapper ScaledModel that automatically scales data during training and prediction.

In [2]:
class ScaledModel(BaseEstimator, ClassifierMixin):
    def __init__(self, base_model):
        self.base_model = base_model
        self.scaler = StandardScaler()
        self.model = None
    
    def fit(self, X, y):
        X_scaled = self.scaler.fit_transform(X)
        self.model = self.base_model
        self.model.fit(X_scaled, y)
        return self
    
    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)
    
    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)
    
    @property
    def classes_(self):
        return self.model.classes_

XGBoost is a powerful boosting algorithm.
If installed, we’ll include it in our model comparisons. Otherwise, we skip it.

In [3]:
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available. Install with: pip install xgboost")


In [4]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


We load the dataset crop_recommendation_with_soil.csv, encode soil types numerically, and split data into training and testing sets.

In [11]:
print("📊 Loading dataset...")

data = pd.read_csv(r"C:\Users\admin\Desktop\major project phase 1\soil_crop_recommender\data\crop_recommendation_with_soil.csv")


print(f"✅ Dataset loaded: {data.shape[0]} samples, {data.shape[1]} features")

print("📄 Preview of dataset:")
print(data.head())
le = LabelEncoder()
data['soil_type_encoded'] = le.fit_transform(data['soil_type'])
print(f"✅ Soil types encoded: {len(le.classes_)} types")

X = data[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'soil_type_encoded']]
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

📊 Loading dataset...
✅ Dataset loaded: 2200 samples, 9 features
📄 Preview of dataset:
    N   P   K  temperature   humidity        ph    rainfall label  \
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice   
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice   
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice   
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice   
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice   

       soil_type  
0  alluvial soil  
1  alluvial soil  
2  alluvial soil  
3  alluvial soil  
4  alluvial soil  
✅ Soil types encoded: 9 types
✅ Train: 1760, Test: 440


We train multiple classifiers individually and compare their accuracies.
Models include Random Forest, Gradient Boosting, Decision Tree, SVM, KNN, Logistic Regression, and optionally XGBoost.

In [13]:
# Encode crop labels for XGBoost compatibility
crop_encoder = LabelEncoder()
y_encoded = crop_encoder.fit_transform(y)

# Use y_encoded for training models that need numeric labels (like XGBoost)


In [ ]:
print("\n Training Multiple Models...")
print("=" * 60)

base_models = {
    'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
}

if XGBOOST_AVAILABLE:
    base_models['XGBoost'] = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

model_scores = {}
trained_models = {}

for name, model in base_models.items():
    print(f"\n🔄 Training {name}...")
    
    if name in ['K-Nearest Neighbors', 'Support Vector Machine', 'Logistic Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
    score = accuracy_score(y_test, y_pred)
    model_scores[name] = score
    trained_models[name] = model
    print(f"✅ {name} Accuracy: {score:.4f} ({score*100:.2f}%)")
